# Nowcasting satellite brightness temperature with flow matching

**Data:** GOES-16 ABI infrared brightness temperature (10.7 µm, the "clean" IR window) from the **SEVIR** benchmark.
Each SEVIR event is a **4-hour sequence of 49 frames, 5 minutes apart**, on a 192 × 192 grid at 2 km (≈ 384 km × 384 km).
Because the sequences are long, we can roll a model forward for **2 hours** and check every step against what the satellite actually saw.

**What you will build**

| Step | Model | Library | Why |
|---|---|---|---|
| 1 | Persistence | – | "Nothing changes" – the baseline to beat |
| 2 | Optical-flow extrapolation | `pysteps` | The classic operational nowcasting method |
| 3 | Deterministic U-Net | `segmentation_models_pytorch` + `lightning` | Learns motion *and* growth/decay, but gets blurry |
| 4 | Residual flow matching | `flow_matching` (Meta) | Adds back sharp, realistic detail and gives an ensemble of possible futures |

Evaluation uses `pysteps` verification tools: CSI for cold cloud tops, CRPS for the ensemble, and power spectra for sharpness.

**Before you start:** *Runtime → Change runtime type → T4 GPU.*
With the default settings the notebook reads about 2.5 GB from a public AWS bucket (no account needed) and trains for roughly 20–30 minutes on a T4.
Set `FAST_DEV_RUN = True` in the configuration cell for a 2-minute smoke test.

## 0. Setup

In [ ]:
!pip -q install lightning segmentation-models-pytorch flow_matching pysteps s3fs h5py

In [ ]:
import math, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

import lightning as L
from lightning.pytorch.loggers import CSVLogger
import segmentation_models_pytorch as smp

from flow_matching.path import CondOTProbPath
from flow_matching.solver import ODESolver
from flow_matching.utils import ModelWrapper

import s3fs
import h5py

from pysteps import motion, nowcasts
from pysteps.utils.spectral import rapsd
from pysteps.verification.detcatscores import det_cat_fct_init, det_cat_fct_accum, det_cat_fct_compute
from pysteps.verification.probscores import CRPS_init, CRPS_accum, CRPS_compute

warnings.filterwarnings("ignore", category=UserWarning)
L.seed_everything(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if USE_AMP else "")
if not USE_AMP:
    print("No GPU found: training will be very slow. Use Runtime -> Change runtime type -> T4 GPU.")

## 1. Configuration

Everything you might want to change is in this cell.

In [ ]:
# ---- Data ----
DATA_DIR = Path("/content/sevir_ir107")   # local cache; point to Google Drive to keep it between sessions
TRAIN_YEAR, TEST_YEAR = "2018", "2019"    # train and test on different years (no leakage)
N_TRAIN_EVENTS = 600      # 4-hour events read for training (~3.6 MB each)
N_TEST_EVENTS = 100       # events read for testing
STRIDE = 2                # SEVIR frames are 5 min apart; STRIDE=2 -> the model steps 10 min at a time

# ---- Task ----
CONTEXT = 3               # past frames the model sees (3 x 10 min = 30 min of history)
LEAD_STEPS = 12           # rollout length for evaluation (12 x 10 min = 2 hours)

# ---- Models / training ----
ENCODER = "resnet18"      # any encoder from segmentation_models_pytorch
BATCH_SIZE = 16
LR = 3e-4
UNET_STEPS = 3000
FM_STEPS = 6000
EMA_DECAY = 0.999

# ---- Sampling / evaluation ----
ODE_STEP_SIZE = 0.125     # midpoint solver with 8 steps from noise (t=0) to data (t=1)
ENSEMBLE_SIZE = 8
EVAL_STARTS = [0, 10]     # forecast start frames inside each test event
COLD_THRESHOLDS_K = [220, 235, 250]   # CSI thresholds: Tb below these = cold (high, deep) cloud tops
TB_CENTER, TB_SCALE = 250.0, 40.0     # normalization: (Tb - 250 K) / 40 K

FAST_DEV_RUN = False      # True = tiny settings, just to check that every cell runs
if FAST_DEV_RUN:
    N_TRAIN_EVENTS, N_TEST_EVENTS = 24, 8
    UNET_STEPS, FM_STEPS = 50, 50
    ENSEMBLE_SIZE, LEAD_STEPS, EVAL_STARTS = 3, 6, [0]

N_FRAMES = (49 - 1) // STRIDE + 1
STEP_MIN = 5 * STRIDE
assert max(EVAL_STARTS) + CONTEXT + LEAD_STEPS <= N_FRAMES, "Rollout does not fit inside a 4-hour event"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"{N_FRAMES} frames per event at {STEP_MIN}-min spacing; "
      f"forecasting {LEAD_STEPS} steps = {LEAD_STEPS * STEP_MIN} min from {CONTEXT} past frames")

## 2. Data: SEVIR IR107 from the public AWS bucket

SEVIR is hosted in the AWS Open Data registry (`s3://sevir`, anonymous access).
IR107 files are HDF5 with an array of shape `(events, 192, 192, 49)` stored as integers in units of 0.01 °C.
We read only the first `N` events of one file per year (h5py reads just those bytes), keep every `STRIDE`-th frame, and cache the result locally.

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

def pick_ir107_file(year):
    files = [f for f in fs.ls(f"sevir/data/ir107/{year}", detail=True) if f["name"].endswith(".h5")]
    if not files:
        raise FileNotFoundError(f"No IR107 files for {year}; try another year (2017-2019).")
    for f in files:
        print(f"   {f['name']}  ({f['size'] / 1e9:.1f} GB)")
    storm = [f for f in files if "STORMEVENTS" in f["name"]]   # storm events = more convection
    return (storm or files)[0]["name"]

def load_events(year, n_events):
    cache = DATA_DIR / f"ir107_{year}_{n_events}_stride{STRIDE}.npy"
    if cache.exists():
        print("Loading cached", cache.name)
        return np.load(cache)
    print(f"IR107 files for {year}:")
    path = pick_ir107_file(year)
    print("Reading from", path)
    t0 = time.time()
    with fs.open(path, "rb", block_size=2**26) as fh, h5py.File(fh, "r") as f:
        key = "ir107" if "ir107" in f else next(k for k in f if getattr(f[k], "ndim", 0) == 4)
        print("   datasets in file:", list(f.keys()), "->", key, f[key].shape, f[key].dtype)
        n = min(n_events, f[key].shape[0])
        parts = []
        for i in range(0, n, 50):                              # read 50 events at a time
            block = f[key][i:min(i + 50, n)]                   # (n, H, W, 49)
            parts.append(np.ascontiguousarray(block[..., ::STRIDE].transpose(0, 3, 1, 2)))
            print(f"\r   {min(i + 50, n)}/{n} events", end="")
        print()
    events = np.concatenate(parts)                             # (N, T, H, W), int16, 0.01 degC
    print(f"   read {events.nbytes / 1e9:.2f} GB in {time.time() - t0:.0f} s")
    np.save(cache, events)
    return events

def quality_filter(events):
    # Drop events with missing/constant frames or physically impossible values.
    keep = []
    for e in events:
        ok = (e.min() > -11000) and (e.max() < 7000) and (e.reshape(e.shape[0], -1).std(axis=1).min() > 0)
        keep.append(ok)
    keep = np.array(keep)
    print(f"   kept {keep.sum()} / {len(keep)} events")
    return events[keep]

train_events = quality_filter(load_events(TRAIN_YEAR, N_TRAIN_EVENTS))
test_events = quality_filter(load_events(TEST_YEAR, N_TEST_EVENTS))
print("train:", train_events.shape, " test:", test_events.shape)

In [ ]:
def raw_to_kelvin(raw):
    return raw.astype(np.float32) * 0.01 + 273.15

def normalize(k):            # Kelvin -> model units
    return (k - TB_CENTER) / TB_SCALE

def to_kelvin(n):            # model units -> Kelvin
    return n * TB_SCALE + TB_CENTER

def raw_to_norm(raw):
    return normalize(raw_to_kelvin(raw)).astype(np.float32)

# Check the units and the time dimension.
k = raw_to_kelvin(train_events[:20])
print(f"Brightness temperature: min {k.min():.1f} K, median {np.median(k):.1f} K, max {k.max():.1f} K")
print(f"Each event has {train_events.shape[1]} frames, {STEP_MIN} min apart = "
      f"{(train_events.shape[1] - 1) * STEP_MIN / 60:.1f} hours")

# Plot one event: cold (high) cloud tops are bright, as on a satellite IR image.
TB_KW = dict(cmap="gray_r", vmin=190, vmax=300)
ev = raw_to_kelvin(train_events[int(np.argmin(k.reshape(len(k), -1).mean(1)))])   # the cloudiest of the 20
show = np.linspace(0, ev.shape[0] - 1, 7).astype(int)
fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 2.8))
for ax, t in zip(axes, show):
    im = ax.imshow(ev[t], **TB_KW); ax.set_title(f"+{t * STEP_MIN} min"); ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="Tb (K)")
plt.show()

### Training windows

A training sample is a window of `CONTEXT + 1` consecutive frames: the model sees the first `CONTEXT` and must predict the last.
Windows are drawn at random positions inside random events, and rotated/flipped for augmentation.
10 % of the training events are held out for validation.

In [ ]:
class RandomWindows(Dataset):
    # Random (event, start) windows of CONTEXT + 1 frames, normalized, with rotation/flip augmentation.
    def __init__(self, events, length, augment=True):
        self.events, self.length, self.augment = events, length, augment
        self.n_events, self.n_t = events.shape[:2]

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        e = int(torch.randint(self.n_events, (1,)))
        s = int(torch.randint(self.n_t - CONTEXT, (1,)))
        x = torch.from_numpy(raw_to_norm(self.events[e, s:s + CONTEXT + 1]))
        if self.augment:
            x = torch.rot90(x, int(torch.randint(4, (1,))), dims=(1, 2))
            if torch.rand(1) < 0.5:
                x = x.flip(-1)
        return x.contiguous()


class FixedWindows(Dataset):
    # Deterministic windows for validation.
    def __init__(self, events, starts_per_event=3):
        n_t = events.shape[1]
        starts = np.linspace(0, n_t - CONTEXT - 1, starts_per_event).astype(int)
        self.events, self.items = events, [(e, s) for e in range(len(events)) for s in starts]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        e, s = self.items[i]
        return torch.from_numpy(raw_to_norm(self.events[e, s:s + CONTEXT + 1]))


n_val = max(2, len(train_events) // 10)
fit_events, val_events = train_events[:-n_val], train_events[-n_val:]
val_loader = DataLoader(FixedWindows(val_events), batch_size=BATCH_SIZE, num_workers=2)

def train_loader(steps):
    ds = RandomWindows(fit_events, length=steps * BATCH_SIZE)
    return DataLoader(ds, batch_size=BATCH_SIZE, num_workers=2, drop_last=True, persistent_workers=True)

print(f"{len(fit_events)} training events, {len(val_events)} validation events, "
      f"{len(val_loader.dataset)} validation windows")

## 3. Deterministic U-Net (segmentation_models_pytorch + Lightning)

The network gets the last `CONTEXT` frames as input channels and predicts the **change** to the most recent frame
(so an untrained network starts close to persistence). It is trained with mean-squared error.

`smp.Unet` gives us a well-tested encoder-decoder; image sides must be multiples of 32 (192 is).

In [ ]:
class ResidualUNet(nn.Module):
    # next_frame = last_frame + UNet(past_frames)
    def __init__(self, in_channels, encoder=ENCODER):
        super().__init__()
        self.net = smp.Unet(encoder_name=encoder, encoder_weights=None, in_channels=in_channels, classes=1)

    def forward(self, past):
        return past[:, -1:] + self.net(past)


def cosine_schedule(module, params):
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=module.trainer.max_steps)
    return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}


class DeterministicNowcaster(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = ResidualUNet(CONTEXT)

    def training_step(self, batch, batch_idx):
        past, target = batch[:, :-1], batch[:, -1:]
        loss = F.mse_loss(self.model(past), target)
        self.log("train_mse", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        past, target = batch[:, :-1], batch[:, -1:]
        self.log("val_mse", F.mse_loss(self.model(past), target), prog_bar=True)
        self.log("val_mse_persistence", F.mse_loss(past[:, -1:], target))

    def configure_optimizers(self):
        return cosine_schedule(self, self.model.parameters())


def make_trainer(max_steps, name):
    return L.Trainer(
        max_steps=max_steps, accelerator="auto", devices=1,
        precision="16-mixed" if USE_AMP else "32-true",
        logger=CSVLogger("/content/logs", name=name),
        val_check_interval=min(500, max_steps), limit_val_batches=20,
        gradient_clip_val=1.0, log_every_n_steps=25, enable_checkpointing=False,
    )


def plot_losses(trainer, train_key, val_keys):
    m = pd.read_csv(Path(trainer.logger.log_dir) / "metrics.csv")
    fig, ax = plt.subplots(figsize=(7, 3.5))
    tr = m.dropna(subset=[train_key])
    ax.plot(tr["step"], tr[train_key].rolling(10, min_periods=1).mean(), label=train_key)
    for k in val_keys:
        if k in m:
            v = m.dropna(subset=[k]); ax.plot(v["step"], v[k], "o-", label=k)
    ax.set_xlabel("step"); ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()

In [ ]:
det_module = DeterministicNowcaster()
det_trainer = make_trainer(UNET_STEPS, "unet")
det_trainer.fit(det_module, train_loader(UNET_STEPS), val_loader)
plot_losses(det_trainer, "train_mse", ["val_mse", "val_mse_persistence"])

unet = det_module.model.to(DEVICE).eval()
for p in unet.parameters():
    p.requires_grad_(False)

The U-Net should beat persistence on validation MSE. Now look at its forecast: MSE training makes it predict the
**average** of all plausible futures, so small cloud features get smeared out. That blur grows when we feed its output back in.

In [ ]:
@torch.no_grad()
def unet_step(past):
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        return unet(past).float()

batch = next(iter(val_loader)).to(DEVICE)
pred = unet_step(batch[:, :-1])
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
panels = [(batch[0, -2], "now (t)"), (batch[0, -1], "truth t+10 min"), (pred[0, 0], "U-Net t+10 min"),
          (pred[0, 0] - batch[0, -1], "U-Net error")]
for ax, (img, title) in zip(axes, panels):
    img = img.cpu().numpy()
    if "error" in title:
        im = ax.imshow(img * TB_SCALE, cmap="RdBu_r", vmin=-20, vmax=20)
    else:
        im = ax.imshow(to_kelvin(img), **TB_KW)
    ax.set_title(title); ax.axis("off"); fig.colorbar(im, ax=ax, shrink=0.75)
plt.show()

## 4. Residual flow matching

We keep the U-Net forecast $\mu$ and learn the **part it cannot predict**, the residual

$$ r = \frac{x_{t+1} - \mu}{\sigma_r}, $$

with a generative model, so that each sample of $r$ gives a sharp, plausible future $x_{t+1} = \mu + \sigma_r\, r$.
This "deterministic backbone + generative correction" design is used by DiffCast, CasCast and CorrDiff
(the best model in Chase et al.'s GOES study).

**Flow matching in four lines** (flow time is called $\tau$ to avoid confusion with physical time):

1. Draw noise $z_0 \sim \mathcal N(0, I)$ and take a real residual $z_1 = r$.
2. Pick $\tau \sim U(0,1)$ and mix them on a straight line: $z_\tau = (1-\tau)\,z_0 + \tau\, z_1$.
3. Train a network $v_\theta(z_\tau, \tau, \text{past}, \mu)$ to predict the direction $z_1 - z_0$ with plain MSE.
4. To forecast, start from fresh noise and follow $v_\theta$ from $\tau = 0$ to $1$ with an ODE solver.

Steps 1–2 are `CondOTProbPath.sample`; step 4 is `ODESolver.sample`, both from Meta's `flow_matching` library.
The velocity network is another `smp.Unet`; $\tau$ is given to it as an extra constant input channel.
We also keep an exponential moving average (EMA) of the weights, which gives cleaner samples.

In [ ]:
# Scale of the residual the generative model has to produce
with torch.no_grad():
    res = [(b[:, -1:].to(DEVICE) - unet_step(b[:, :-1].to(DEVICE))).cpu() for b in val_loader]
SIGMA_R = torch.cat(res).std().item()
print(f"Residual std: {SIGMA_R:.4f} (model units) = {SIGMA_R * TB_SCALE:.2f} K")


def velocity(net, z, tau, past, mu):
    tau_map = tau.view(-1, 1, 1, 1).expand(-1, 1, *z.shape[-2:])
    return net(torch.cat([z, past, mu, tau_map], dim=1))


class FlowMatchingNowcaster(L.LightningModule):
    def __init__(self, unet, sigma_r):
        super().__init__()
        self.unet = unet                       # frozen deterministic backbone
        self.sigma_r = sigma_r
        self.net = smp.Unet(encoder_name=ENCODER, encoder_weights=None,
                            in_channels=1 + CONTEXT + 1 + 1, classes=1)   # z, past frames, mu, tau
        self.ema = AveragedModel(self.net, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY))
        self.ema.requires_grad_(False)
        self.path = CondOTProbPath()

    def fm_loss(self, batch, net):
        past, target = batch[:, :-1], batch[:, -1:]
        with torch.no_grad():
            mu = self.unet(past)
        r = (target - mu) / self.sigma_r
        z0 = torch.randn_like(r)
        tau = torch.rand(r.shape[0], device=r.device)
        ps = self.path.sample(x_0=z0, x_1=r, t=tau)          # ps.x_t on the line, ps.dx_t = r - z0
        return F.mse_loss(velocity(net, ps.x_t, tau, past, mu), ps.dx_t)

    def on_train_batch_start(self, batch, batch_idx):
        self.unet.eval()                       # keep the backbone's BatchNorm statistics frozen

    def training_step(self, batch, batch_idx):
        loss = self.fm_loss(batch, self.net)
        self.log("train_fm_loss", loss, prog_bar=True)
        return loss

    def on_train_batch_end(self, outputs, batch, batch_idx):
        self.ema.update_parameters(self.net)

    def validation_step(self, batch, batch_idx):
        self.log("val_fm_loss", self.fm_loss(batch, self.net), prog_bar=True)

    def configure_optimizers(self):
        return cosine_schedule(self, self.net.parameters())

    def ema_net(self):
        # EMA weights + the live network's BatchNorm statistics, ready for sampling
        for b_ema, b in zip(self.ema.module.buffers(), self.net.buffers()):
            b_ema.copy_(b)
        return self.ema.module.eval()

In [ ]:
fm_module = FlowMatchingNowcaster(unet, SIGMA_R)
fm_trainer = make_trainer(FM_STEPS, "flow_matching")
fm_trainer.fit(fm_module, train_loader(FM_STEPS), val_loader)
plot_losses(fm_trainer, "train_fm_loss", ["val_fm_loss"])
fm_module = fm_module.to(DEVICE)

### Sampling and autoregressive rollout

`ODESolver` integrates the learned velocity field from noise to a residual sample.
For a multi-step forecast we feed each predicted frame back in as input (an *autoregressive rollout*).
Each ensemble member keeps its own trajectory, so the members can drift apart as uncertainty grows, the same way GenCast builds its weather ensembles.

In [ ]:
class VelocityField(ModelWrapper):
    # Adapts our network to the (x, t, **extras) call used by flow_matching's ODESolver.
    def forward(self, x, t, **extras):
        if t.numel() == 1:
            t = t.reshape(1).to(x).expand(x.shape[0])
        with torch.autocast(device_type=x.device.type, dtype=torch.float16, enabled=x.device.type == "cuda"):
            v = velocity(self.model, x, t, extras["past"], extras["mu"])
        return v.float()


solver = ODESolver(velocity_model=VelocityField(fm_module.ema_net()))

@torch.no_grad()
def rollout(past, steps, method, members=1):
    # past: (B, CONTEXT, H, W) normalized -> (members, B, steps, H, W) normalized
    B, H, W = past.shape[0], past.shape[-2], past.shape[-1]
    state = past.repeat(members, 1, 1, 1)
    frames = []
    for _ in range(steps):
        mu = unet_step(state)
        if method == "unet":
            nxt = mu
        else:
            z = solver.sample(x_init=torch.randn_like(mu), step_size=ODE_STEP_SIZE, method="midpoint",
                              time_grid=torch.tensor([0.0, 1.0], device=mu.device), past=state, mu=mu)
            nxt = mu + SIGMA_R * z
        frames.append(nxt)
        state = torch.cat([state[:, 1:], nxt], dim=1)
    return torch.cat(frames, dim=1).view(members, B, steps, H, W)

# Quick look: three possible futures for one validation window, 10 min ahead
batch = next(iter(val_loader))[:1].to(DEVICE)
ens = rollout(batch[:, :-1], 1, "fm", members=3)[:, 0, 0].cpu().numpy()
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
panels = [(batch[0, -1].cpu().numpy(), "truth t+10 min"),
          (unet_step(batch[:, :-1])[0, 0].cpu().numpy(), "U-Net")] + [(ens[i], f"FM member {i + 1}") for i in range(3)]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(to_kelvin(img), **TB_KW); ax.set_title(title); ax.axis("off")
plt.show()

## 5. Multi-step evaluation on held-out events (2 hours ahead)

For every test window we forecast `LEAD_STEPS` steps from `CONTEXT` observed frames and compare each step with the real frames.

* **Persistence**: repeat the last observed frame.
* **Optical flow**: `pysteps` Lucas–Kanade motion field, then semi-Lagrangian extrapolation of the last frame.
* **U-Net**: deterministic rollout.
* **Flow matching**: `ENSEMBLE_SIZE` members; we score one member and the ensemble mean.

**Metrics**

* **RMSE** (K): penalises sharp forecasts that put a feature slightly in the wrong place twice (the *double penalty*), so blurry forecasts look good on it.
* **CSI** for Tb below 220/235/250 K: how well cold cloud-top areas are located (1 = perfect).
* **CRPS** (K): the probabilistic version of MAE. For a single deterministic forecast it equals the MAE, so the U-Net MAE and the ensemble CRPS can be compared directly.
* **Spread/skill**: ensemble spread divided by the RMSE of the ensemble mean (≈ 1 means the spread honestly reflects the error).
* **Power spectrum** at +60 min: blurry forecasts lose power at small scales.

In [ ]:
LK = motion.get_method("LK")
EXTRAPOLATE = nowcasts.get_method("extrapolation")
_flow_failures = [0]

def optical_flow_forecast(past_k):
    # past_k: (CONTEXT, H, W) Kelvin -> (LEAD_STEPS, H, W) Kelvin
    try:
        cloudiness = np.clip(320.0 - past_k, 0, None)    # cold clouds become large values, like rain rate
        V = LK(cloudiness, verbose=False)
        fc = EXTRAPOLATE(past_k[-1], V, LEAD_STEPS)
        return np.where(np.isfinite(fc), fc, past_k[-1][None]).astype(np.float32)
    except Exception as err:
        if _flow_failures[0] == 0:
            print("Optical flow failed, using persistence for this case:", repr(err))
        _flow_failures[0] += 1
        return np.repeat(past_k[-1:], LEAD_STEPS, axis=0)

METHODS = ["Persistence", "Optical flow", "U-Net", "FM (1 member)", "FM (ens. mean)"]
LEADS_MIN = (np.arange(LEAD_STEPS) + 1) * STEP_MIN
SPEC_LEAD = min(LEAD_STEPS, 60 // STEP_MIN) - 1          # index of the +60 min lead

sq_err = {m: np.zeros(LEAD_STEPS) for m in METHODS}
csi_tab = {m: {thr: [det_cat_fct_init(-thr) for _ in range(LEAD_STEPS)] for thr in COLD_THRESHOLDS_K} for m in METHODS}
mae_unet = np.zeros(LEAD_STEPS)
crps_tab = [CRPS_init() for _ in range(LEAD_STEPS)]
spread_sum = np.zeros(LEAD_STEPS)
n_pix = 0
spectra = {m: [] for m in METHODS + ["Observed"]}

windows = [(e, s) for e in range(len(test_events)) for s in EVAL_STARTS]
EVAL_BATCH = 8
t0 = time.time()
for i in range(0, len(windows), EVAL_BATCH):
    chunk = windows[i:i + EVAL_BATCH]
    win = np.stack([raw_to_norm(test_events[e, s:s + CONTEXT + LEAD_STEPS]) for e, s in chunk])
    past = torch.from_numpy(win[:, :CONTEXT]).to(DEVICE)
    truth = to_kelvin(win[:, CONTEXT:])                                      # (B, LEAD, H, W)
    ens = to_kelvin(rollout(past, LEAD_STEPS, "fm", ENSEMBLE_SIZE).cpu().numpy())   # (M, B, LEAD, H, W)
    preds = {
        "Persistence": np.repeat(to_kelvin(win[:, CONTEXT - 1:CONTEXT]), LEAD_STEPS, axis=1),
        "Optical flow": np.stack([optical_flow_forecast(to_kelvin(w[:CONTEXT])) for w in win]),
        "U-Net": to_kelvin(rollout(past, LEAD_STEPS, "unet")[0].cpu().numpy()),
        "FM (1 member)": ens[0],
        "FM (ens. mean)": ens.mean(0),
    }
    for m, p in preds.items():
        sq_err[m] += ((p - truth) ** 2).sum(axis=(0, 2, 3))
        for thr in COLD_THRESHOLDS_K:
            for l in range(LEAD_STEPS):
                det_cat_fct_accum(csi_tab[m][thr][l], -p[:, l], -truth[:, l])   # minus sign: "below thr" = event
    mae_unet += np.abs(preds["U-Net"] - truth).sum(axis=(0, 2, 3))
    spread_sum += ens.var(axis=0, ddof=1).sum(axis=(0, 2, 3))
    for b in range(len(chunk)):
        for l in range(LEAD_STEPS):
            CRPS_accum(crps_tab[l], ens[:, b, l], truth[b, l])
        spectra["Observed"].append(rapsd(truth[b, SPEC_LEAD], fft_method=np.fft, return_freq=True))
        for m, p in preds.items():
            spectra[m].append(rapsd(p[b, SPEC_LEAD], fft_method=np.fft, return_freq=True))
    n_pix += truth.shape[0] * truth.shape[2] * truth.shape[3]
    done = i + len(chunk)
    print(f"\r{done}/{len(windows)} windows, {time.time() - t0:.0f} s", end="")
print(f"\nOptical-flow fallbacks: {_flow_failures[0]}")

rmse = {m: np.sqrt(sq_err[m] / n_pix) for m in METHODS}
csi = {m: {thr: np.array([det_cat_fct_compute(c, scores="CSI")["CSI"] for c in csi_tab[m][thr]])
           for thr in COLD_THRESHOLDS_K} for m in METHODS}
crps_fm = np.array([CRPS_compute(c) for c in crps_tab])
mae_unet = mae_unet / n_pix
spread_skill = np.sqrt(spread_sum / n_pix) / rmse["FM (ens. mean)"]

In [ ]:
CSI_THR = 235 if 235 in COLD_THRESHOLDS_K else COLD_THRESHOLDS_K[0]
pick = sorted({i for i in (0, 60 // STEP_MIN - 1, LEAD_STEPS - 1) if 0 <= i < LEAD_STEPS})
rows = {}
for m in METHODS:
    row = {f"RMSE {LEADS_MIN[i]} min (K)": rmse[m][i] for i in pick}
    row.update({f"CSI<{CSI_THR}K {LEADS_MIN[i]} min": csi[m][CSI_THR][i] for i in pick})
    rows[m] = row
rows["U-Net"].update({f"CRPS {LEADS_MIN[i]} min (K)": mae_unet[i] for i in pick})
rows["FM (ens. mean)"].update({f"CRPS {LEADS_MIN[i]} min (K)": crps_fm[i] for i in pick})
display(pd.DataFrame(rows).T.round(3))

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for m in METHODS:
    axes[0].plot(LEADS_MIN, rmse[m], "o-", ms=3, label=m)
    axes[1].plot(LEADS_MIN, csi[m][CSI_THR], "o-", ms=3, label=m)
axes[0].set(title="RMSE (lower is better)", ylabel="K")
axes[1].set(title=f"CSI, Tb < {CSI_THR} K (higher is better)")
axes[2].plot(LEADS_MIN, mae_unet, "o-", ms=3, label="U-Net (MAE = CRPS)")
axes[2].plot(LEADS_MIN, crps_fm, "o-", ms=3, label=f"FM ensemble ({ENSEMBLE_SIZE} members)")
axes[2].set(title="CRPS (lower is better)", ylabel="K")
axes[3].plot(LEADS_MIN, spread_skill, "o-", ms=3, color="k")
axes[3].axhline(1, ls="--", color="gray")
axes[3].set(title="Spread / skill (1 = well calibrated)")
for ax in axes:
    ax.set_xlabel("lead time (min)"); ax.grid(alpha=0.3)
for ax in axes[:3]:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for m in ["Observed"] + METHODS:
    power = np.mean([s for s, f in spectra[m]], axis=0)
    freq = spectra[m][0][1]
    ok = freq > 0
    ax.loglog(2.0 / freq[ok], power[ok], lw=2.5 if m == "Observed" else 1.5,
              color="k" if m == "Observed" else None, label=m)
ax.invert_xaxis()
ax.set(xlabel="wavelength (km)", ylabel="power", title=f"Power spectrum at +{LEADS_MIN[SPEC_LEAD]} min")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.show()

**How to read the results**

* At short lead times persistence and optical flow are hard to beat: clouds barely change in 10 minutes.
* The U-Net usually has the best RMSE of the single forecasts, but its spectrum drops at small wavelengths: it is blurry, and gets blurrier with every rollout step.
* A single flow-matching member often has a *worse* RMSE than the U-Net (double penalty) but a spectrum close to the observed one: it looks like real satellite imagery.
* The ensemble mean recovers a good RMSE, and the CRPS is the fair way to compare a probabilistic forecast with a deterministic one.
* A spread/skill ratio below 1 means the ensemble is over-confident (common with few members or short training).

Check whether your numbers follow this pattern; with the small default training budget they may not all hold.

In [ ]:
# Case study: the test window with the most cold cloud at +60 min
cold = [(raw_to_kelvin(test_events[e, s + CONTEXT + SPEC_LEAD]) < 235).mean() for e, s in windows]
e, s = windows[int(np.argmax(cold))]
win = raw_to_norm(test_events[e, s:s + CONTEXT + LEAD_STEPS])
past = torch.from_numpy(win[None, :CONTEXT]).to(DEVICE)
truth = to_kelvin(win[CONTEXT:])
ens = to_kelvin(rollout(past, LEAD_STEPS, "fm", members=2)[:, 0].cpu().numpy())
rows = {
    "Observed": truth,
    "Persistence": np.repeat(to_kelvin(win[CONTEXT - 1:CONTEXT]), LEAD_STEPS, axis=0),
    "Optical flow": optical_flow_forecast(to_kelvin(win[:CONTEXT])),
    "U-Net": to_kelvin(rollout(past, LEAD_STEPS, "unet")[0, 0].cpu().numpy()),
    "FM member 1": ens[0],
    "FM member 2": ens[1],
}
cols = sorted(set(np.linspace(0, LEAD_STEPS - 1, 5).astype(int)))
fig, axes = plt.subplots(len(rows), len(cols), figsize=(2.6 * len(cols), 2.6 * len(rows)))
for r, (name, seq) in enumerate(rows.items()):
    for c, l in enumerate(cols):
        ax = axes[r, c]
        ax.imshow(seq[l], **TB_KW); ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(f"+{LEADS_MIN[l]} min")
        if c == 0: ax.set_ylabel(name)
plt.tight_layout(); plt.show()

## 6. Save the models

In [ ]:
torch.save({
    "unet": unet.state_dict(),
    "fm_ema": fm_module.ema_net().state_dict(),
    "sigma_r": SIGMA_R,
    "config": dict(CONTEXT=CONTEXT, STRIDE=STRIDE, ENCODER=ENCODER, TB_CENTER=TB_CENTER, TB_SCALE=TB_SCALE),
}, DATA_DIR / "ir107_flow_matching_nowcaster.pt")
print("Saved to", DATA_DIR / "ir107_flow_matching_nowcaster.pt")

## 7. Exercises

1. **More data and training.** Increase `N_TRAIN_EVENTS` and `FM_STEPS`. Which metric improves most?
2. **Sampling budget.** Try `ODE_STEP_SIZE` of 0.5, 0.25 and 0.0625, and the `"euler"` vs `"midpoint"` methods. How many steps does flow matching really need?
3. **Ensemble size.** Compare CRPS and spread/skill for 2, 4, 8 and 16 members.
4. **Longer memory.** Change `CONTEXT` to 1, 3 and 6. Does more history help at 2 hours?
5. **Diffusion vs flow matching.** Replace `CondOTProbPath` with a diffusion-style path from the same library (`AffineProbPath` with a `VPScheduler`) and compare.
6. **Other channels.** SEVIR also has the 6.9 µm water-vapour channel (`ir069`). Add it as extra input channels.
7. **Train on the full year.** SEVIR also includes `RANDOMEVENTS` files; mixing them in reduces the storm bias.

## References

* Veillette, Samsi & Mattioli (2020). *SEVIR: A Storm Event Imagery Dataset for Deep Learning Applications in Radar and Satellite Meteorology.* NeurIPS. Data: `s3://sevir` (AWS Open Data).
* Lipman et al. (2023). *Flow Matching for Generative Modeling.* ICLR; library: `facebookresearch/flow_matching`.
* Chase, Haynes, Ver Hoef & Ebert-Uphoff (2026). *How to use score-based diffusion in earth system science: A satellite nowcasting example.* AIES (arXiv:2505.10432).
* Yu et al. (2024). *DiffCast: A Unified Framework via Residual Diffusion for Precipitation Nowcasting.* CVPR.
* Pulkkinen et al. (2019). *Pysteps: an open-source Python library for probabilistic precipitation nowcasting.* GMD.